# ADD_STUDENT_DEGREE_STATUS clean notebook

This notebook cleans `ADD_STUDENT_DEGREE_STATUS` for academic advice and course recommendation work.

It assumes `df_raw` is already loaded. It does not read from a database, read parquet files, access credentials, train models, split data, or save parquet output.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.paths import RAW_DIR
from src.cleaning_utils import integer_like_report, normalize_id_columns

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

sns.set_theme(style="whitegrid")

## Table role and grain

`ADD_STUDENT_DEGREE_STATUS` stores one academic status row per student per semester.

Primary key: `student_status_id`.

Logical key: `student_id` + `part_id`.

`finish_status` and `finish_part_id` are kept for audit, filtering, and later outcome construction. They must not be used later as model input features because they describe final path-level outcomes and can leak future information.

In [ ]:
df_raw = pd.read_parquet(RAW_DIR / "v_add_student_degree_status.parquet")
assert "df_raw" in globals(), "df_raw must already be loaded before running this notebook."

df = df_raw.copy()

normalized_columns = (
    pd.Index(df.columns)
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "_", regex=True)
    .str.replace(r"[^0-9a-zA-Z_]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)

if normalized_columns.duplicated().any():
    duplicate_columns = normalized_columns[normalized_columns.duplicated()].tolist()
    raise ValueError(f"Column-name normalization created duplicate column names: {duplicate_columns}")

df.columns = normalized_columns

initial_rows = len(df)
print(f"Initial shape: {df.shape}")

In [ ]:
df_raw.info()

In [ ]:
REQUIRED_COLUMNS = [
    "student_status_id",
    "student_id",
    "part_id",
    "degree_id",
    "start_part_id",
    "finish_part_id",
    "grade_version_id",
    "study_mode",
    "permanent_status_id",
    "prev_gpa_points",
    "prev_gpa_percent",
    "gpa_percent",
    "gpa_points",
    "start_agpa_percent",
    "start_agpa_points",
    "start_total_in_courses",
    "start_total_in_credits",
    "end_total_in_courses",
    "end_total_in_credits",
    "end_agpa_percent",
    "end_agpa_points",
    "semester_reg_courses",
    "semester_reg_credits",
    "semester_pass_courses",
    "semester_pass_credits",
    "semester_fail_courses",
    "semester_fail_credits",
    "semester_in_courses",
    "semester_in_credits",
    "total_semesters",
    "total_reg_courses",
    "total_reg_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
    "reg_total_semesters",
    "finish_status",
    "version_title_sl",
    "degree_name_sl",
    "degree_credits_count",
    "start_level_id",
    "start_level_name_pl",
]

missing_columns = [column for column in REQUIRED_COLUMNS if column not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print(f"All required columns are present: {len(REQUIRED_COLUMNS)} columns")

## ID handling

Entity IDs can carry source-system suffixes such as `12345.111`, so they are normalized with the existing `src.cleaning_utils.normalize_id_columns` function and kept as strings.

Semester codes (`part_id`, `start_part_id`, `finish_part_id`) are not suffix-normalized. They are validated as integer-like semester codes and stored as nullable integers.

In [ ]:
entity_id_columns = [
    "student_status_id",
    "student_id",
    "degree_id",
    "grade_version_id",
    "start_level_id",
]

semester_code_columns = [
    "part_id",
    "start_part_id",
    "finish_part_id",
]

status_code_columns = ["permanent_status_id"]

id_validation_report = pd.DataFrame(
    [integer_like_report(df, column) for column in entity_id_columns + semester_code_columns + status_code_columns]
)

print("ID and code validation report before conversion:")
display(id_validation_report)

df = normalize_id_columns(df, entity_id_columns)

for column in semester_code_columns + status_code_columns:
    numeric = pd.to_numeric(df[column], errors="coerce")
    source_has_value = df[column].notna() & df[column].astype("string").str.strip().ne("")

    invalid_values = source_has_value & numeric.isna()
    if invalid_values.any():
        examples = df.loc[invalid_values, column].drop_duplicates().head(10).tolist()
        raise ValueError(f"{column} contains non-numeric values. Examples: {examples}")

    non_integer_values = numeric.notna() & ~np.isclose(numeric % 1, 0, atol=1e-9)
    if non_integer_values.any():
        examples = df.loc[non_integer_values, column].drop_duplicates().head(10).tolist()
        raise ValueError(f"{column} contains non-integer values. Examples: {examples}")

    df[column] = numeric.round().astype("Int64")

id_dtype_report_after_cleaning = pd.DataFrame(
    {
        "column": entity_id_columns + semester_code_columns + status_code_columns,
        "dtype_after_cleaning": [str(df[column].dtype) for column in entity_id_columns + semester_code_columns + status_code_columns],
        "null_count_after_cleaning": [int(df[column].isna().sum()) for column in entity_id_columns + semester_code_columns + status_code_columns],
    }
)

print("ID and code dtypes after cleaning:")
display(id_dtype_report_after_cleaning)

In [ ]:
text_columns = [
    "finish_status",
    "version_title_sl",
    "degree_name_sl",
    "start_level_name_pl",
]

for column in text_columns:
    cleaned = df[column].astype("string").str.strip()
    empty_like = cleaned.eq("") | cleaned.str.lower().isin(["nan", "none", "null", "<na>"])
    df[column] = cleaned.mask(empty_like, pd.NA)

numeric_columns = [
    "prev_gpa_points",
    "prev_gpa_percent",
    "gpa_points",
    "gpa_percent",
    "start_agpa_points",
    "start_agpa_percent",
    "end_agpa_points",
    "end_agpa_percent",
    "semester_reg_courses",
    "semester_reg_credits",
    "semester_pass_courses",
    "semester_pass_credits",
    "semester_fail_courses",
    "semester_fail_credits",
    "semester_in_courses",
    "semester_in_credits",
    "start_total_in_courses",
    "start_total_in_credits",
    "end_total_in_courses",
    "end_total_in_credits",
    "total_semesters",
    "reg_total_semesters",
    "total_reg_courses",
    "total_reg_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
    "degree_credits_count",
]

for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce").astype("Float64")

In [ ]:
duplicate_student_status_count = int(df["student_status_id"].duplicated().sum())
duplicate_logical_key_count = int(df.duplicated(["student_id", "part_id"]).sum())

print(f"student_status_id is unique: {df['student_status_id'].is_unique}")
print(f"Duplicate student_status_id rows: {duplicate_student_status_count}")
print(f"Duplicate student_id + part_id rows: {duplicate_logical_key_count}")

duplicate_student_status_report = df.loc[df["student_status_id"].duplicated(keep=False)].sort_values("student_status_id")
duplicate_logical_key_report = df.loc[df.duplicated(["student_id", "part_id"], keep=False)].sort_values(["student_id", "part_id"])

if duplicate_student_status_count > 0:
    print("Duplicate student_status_id rows:")
    display(duplicate_student_status_report)

if duplicate_logical_key_count > 0:
    print("Duplicate student_id + part_id rows:")
    display(duplicate_logical_key_report)

if duplicate_student_status_count > 0 or duplicate_logical_key_count > 0:
    raise ValueError("Duplicate key values found. This violates the expected table grain.")

In [ ]:
df['student_id'].nunique()

In [ ]:
initial_null_report = (
    pd.DataFrame(
        {
            "column": df.columns,
            "dtype": [str(dtype) for dtype in df.dtypes],
            "null_count": [int(df[column].isna().sum()) for column in df.columns],
            "null_percent": [round(float(df[column].isna().mean() * 100), 4) for column in df.columns],
        }
    )
    .sort_values(["null_count", "column"], ascending=[False, True])
    .reset_index(drop=True)
)

initial_null_report

## Cleaning rules

Preprocessing steps:

1. Drop `permanent_status_id` administrative incomplete registrations.
2. Drop `degree_id` null rows.
3. Drop `WITHDRAWN` students with one zero-activity record.
4. Drop core academic null rows.
5. Drop `study_mode` column.
6. Create `df_clean`.

Dropped rows:

- incomplete administrative registrations identified by `permanent_status_id`
- rows without `degree_id`
- `WITHDRAWN` students with exactly one row and no registered, passed, failed, GPA, or AGPA activity
- rows with nulls in the core academic columns

Intentionally kept:

- graduated students
- active/current students where `finish_status` is null
- withdrawn students with more than one record or any real academic activity
- close-file, cancel-admission, change-plan, change-degree, change-faculty, final-dismiss, and other remaining final statuses
- first-semester nulls in `prev_gpa_points` and `prev_gpa_percent`
- null `finish_part_id`
- `start_level_id` nulls when `start_level_name_pl` is available

In [ ]:
EXCLUDED_PERMANENT_STATUS_IDS = [1, 4, 11, 12, 15, 16, 41]

CORE_ACADEMIC_COLUMNS = [
    "grade_version_id",
    "gpa_percent",
    "gpa_points",
    "start_agpa_percent",
    "start_agpa_points",
    "end_agpa_percent",
    "end_agpa_points",
    "semester_reg_courses",
    "semester_reg_credits",
    "semester_pass_courses",
    "semester_pass_credits",
    "semester_fail_courses",
    "semester_fail_credits",
    "semester_in_courses",
    "semester_in_credits",
    "total_semesters",
    "total_reg_courses",
    "total_reg_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
    "version_title_sl",
]

WITHDRAWN_ZERO_ACTIVITY_COLUMNS = [
    "gpa_points",
    "gpa_percent",
    "end_agpa_points",
    "end_agpa_percent",
    "total_reg_courses",
    "total_reg_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
]

FINAL_COLUMNS = [
    "student_status_id",
    "student_id",
    "part_id",
    "degree_id",
    "start_part_id",
    "finish_part_id",
    "grade_version_id",
    "prev_gpa_points",
    "prev_gpa_percent",
    "gpa_points",
    "gpa_percent",
    "start_agpa_points",
    "start_agpa_percent",
    "end_agpa_points",
    "end_agpa_percent",
    "semester_reg_courses",
    "semester_reg_credits",
    "semester_pass_courses",
    "semester_pass_credits",
    "semester_fail_courses",
    "semester_fail_credits",
    "semester_in_courses",
    "semester_in_credits",
    "start_total_in_courses",
    "start_total_in_credits",
    "end_total_in_courses",
    "end_total_in_credits",
    "total_semesters",
    "reg_total_semesters",
    "total_reg_courses",
    "total_reg_credits",
    "total_pass_courses",
    "total_pass_credits",
    "total_fail_courses",
    "total_fail_credits",
    "finish_status",
    "version_title_sl",
    "degree_name_sl",
    "degree_credits_count",
    "start_level_id",
    "start_level_name_pl",
]

In [ ]:
rows_before_permanent_status = len(df)
permanent_status_drop_mask = df["permanent_status_id"].isin(EXCLUDED_PERMANENT_STATUS_IDS)

dropped_permanent_status_report = df.loc[permanent_status_drop_mask].copy()
df = df.loc[~permanent_status_drop_mask].copy()

rows_dropped_permanent_status = len(dropped_permanent_status_report)
rows_after_permanent_status_filter = len(df)

print(f"Rows before permanent_status_id filter: {rows_before_permanent_status}")
print(f"Rows dropped by permanent_status_id filter: {rows_dropped_permanent_status}")
print(f"Rows remaining after permanent_status_id filter: {rows_after_permanent_status_filter}")

In [ ]:
rows_before_degree_null_drop = len(df)
degree_null_drop_mask = df["degree_id"].isna()

dropped_degree_null_report = df.loc[degree_null_drop_mask].copy()
df = df.loc[~degree_null_drop_mask].copy()

rows_dropped_degree_null = len(dropped_degree_null_report)
rows_after_degree_null_drop = len(df)

print(f"Rows before degree_id null drop: {rows_before_degree_null_drop}")
print(f"Rows dropped by degree_id null: {rows_dropped_degree_null}")
print(f"Rows remaining after degree_id null drop: {rows_after_degree_null_drop}")

In [ ]:
rows_before_withdrawn_single_zero_activity_drop = len(df)
student_row_counts = df.groupby("student_id", dropna=False)["student_id"].transform("size")
zero_activity_mask = df[WITHDRAWN_ZERO_ACTIVITY_COLUMNS].eq(0).fillna(False).all(axis=1)
withdrawn_single_zero_activity_drop_mask = (
    df["finish_status"].eq("WITHDRAWN").fillna(False)
    & student_row_counts.eq(1)
    & zero_activity_mask
)

dropped_withdrawn_single_zero_activity_report = df.loc[withdrawn_single_zero_activity_drop_mask].copy()
df = df.loc[~withdrawn_single_zero_activity_drop_mask].copy()

rows_dropped_withdrawn_single_zero_activity = len(dropped_withdrawn_single_zero_activity_report)
rows_after_withdrawn_single_zero_activity_drop = len(df)

print(f"Rows before withdrawn single zero-activity drop: {rows_before_withdrawn_single_zero_activity_drop}")
print(f"Rows dropped by withdrawn single zero-activity rule: {rows_dropped_withdrawn_single_zero_activity}")
print(f"Rows remaining after withdrawn single zero-activity drop: {rows_after_withdrawn_single_zero_activity_drop}")

In [ ]:
rows_before_core_null_drop = len(df)
core_null_drop_mask = df[CORE_ACADEMIC_COLUMNS].isna().any(axis=1)

dropped_core_null_report = df.loc[core_null_drop_mask].copy()
df = df.loc[~core_null_drop_mask].copy()

rows_dropped_core_null = len(dropped_core_null_report)
rows_after_core_academic_null_drop = len(df)

print(f"Rows before core academic null drop: {rows_before_core_null_drop}")
print(f"Rows dropped by core academic nulls: {rows_dropped_core_null}")
print(f"Rows remaining after core academic null drop: {rows_after_core_academic_null_drop}")

In [ ]:
df = df.drop(columns=["study_mode"], errors="ignore")

print("Dropped study_mode column.")

In [ ]:
df.head()

In [ ]:
df['start_level_name_pl'].value_counts()

In [ ]:
df_clean = df[FINAL_COLUMNS].copy()
final_rows = len(df_clean)

print(f"Final clean shape: {df_clean.shape}")

In [ ]:
duplicate_final_student_status_count = int(df_clean["student_status_id"].duplicated().sum())
duplicate_final_logical_key_count = int(df_clean.duplicated(["student_id", "part_id"]).sum())

if duplicate_final_student_status_count > 0:
    display(df_clean.loc[df_clean["student_status_id"].duplicated(keep=False)].sort_values("student_status_id"))
    raise ValueError(f"student_status_id must be unique. Duplicate row count: {duplicate_final_student_status_count}")

if duplicate_final_logical_key_count > 0:
    display(df_clean.loc[df_clean.duplicated(["student_id", "part_id"], keep=False)].sort_values(["student_id", "part_id"]))
    raise ValueError(f"student_id + part_id must be unique. Duplicate row count: {duplicate_final_logical_key_count}")

assert df_clean["degree_id"].notna().all(), "degree_id must have no nulls."
assert df_clean[CORE_ACADEMIC_COLUMNS].notna().all().all(), "Core academic columns must have no nulls."
assert "study_mode" not in df_clean.columns, "study_mode must not exist in df_clean."
assert "permanent_status_id" not in df_clean.columns, "permanent_status_id must not exist in df_clean."
assert list(df_clean.columns) == FINAL_COLUMNS, "df_clean does not contain exactly the selected final columns in order."

print("Final validation checks passed.")

In [ ]:
cleaning_summary = pd.DataFrame(
    {
        "metric": [
            "initial_rows",
            "rows_dropped_by_permanent_status_id",
            "rows_dropped_by_degree_id_null",
            "rows_dropped_by_withdrawn_single_zero_activity",
            "rows_dropped_by_core_academic_nulls",
            "final_rows",
            "final_columns_count",
        ],
        "value": [
            initial_rows,
            rows_dropped_permanent_status,
            rows_dropped_degree_null,
            rows_dropped_withdrawn_single_zero_activity,
            rows_dropped_core_null,
            final_rows,
            len(df_clean.columns),
        ],
    }
)

final_finish_status_distribution = df_clean["finish_status"].value_counts(dropna=False).rename("row_count").to_frame()

final_null_report = (
    pd.DataFrame(
        {
            "column": df_clean.columns,
            "dtype": [str(dtype) for dtype in df_clean.dtypes],
            "null_count": [int(df_clean[column].isna().sum()) for column in df_clean.columns],
            "null_percent": [round(float(df_clean[column].isna().mean() * 100), 4) for column in df_clean.columns],
        }
    )
    .sort_values(["null_count", "column"], ascending=[False, True])
    .reset_index(drop=True)
)

print("Cleaning summary:")
display(cleaning_summary)

print("Final finish_status distribution after cleaning:")
display(final_finish_status_distribution)

print("Final null report after cleaning:")
display(final_null_report)

## Simple validation plots

These plots are only for cleaning validation and basic data understanding. They do not create model features.

In [ ]:
row_count_plot_data = pd.DataFrame(
    {
        "stage": [
            "initial rows",
            "after permanent status filter",
            "after degree null drop",
            "after withdrawn zero-activity drop",
            "after core null drop",
            "final clean rows",
        ],
        "row_count": [
            initial_rows,
            rows_after_permanent_status_filter,
            rows_after_degree_null_drop,
            rows_after_withdrawn_single_zero_activity_drop,
            rows_after_core_academic_null_drop,
            final_rows,
        ],
    }
)

fig, ax = plt.subplots(figsize=(12, 5))
sns.barplot(data=row_count_plot_data, x="stage", y="row_count", color="#4C78A8", ax=ax)
ax.set_title("Row Count Before and After Cleaning")
ax.set_xlabel("Cleaning stage")
ax.set_ylabel("Row count")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
dropped_rows_plot_data = pd.DataFrame(
    {
        "reason": [
            "dropped_permanent_status",
            "dropped_degree_null",
            "dropped_withdrawn_single_zero_activity",
            "dropped_core_null",
        ],
        "row_count": [
            rows_dropped_permanent_status,
            rows_dropped_degree_null,
            rows_dropped_withdrawn_single_zero_activity,
            rows_dropped_core_null,
        ],
    }
)

fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=dropped_rows_plot_data, x="reason", y="row_count", color="#F58518", ax=ax)
ax.set_title("Dropped Rows by Reason")
ax.set_xlabel("Drop reason")
ax.set_ylabel("Row count")
plt.xticks(rotation=25, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
finish_status_plot_data = (
    df_clean["finish_status"]
    .fillna("NULL / ACTIVE")
    .astype("string")
    .value_counts(dropna=False)
    .rename_axis("finish_status")
    .reset_index(name="row_count")
)

fig, ax = plt.subplots(figsize=(10, 6))
sns.barplot(data=finish_status_plot_data, y="finish_status", x="row_count", color="#54A24B", ax=ax)
ax.set_title("Finish Status Distribution After Cleaning")
ax.set_xlabel("Row count")
ax.set_ylabel("Finish status")
plt.tight_layout()
plt.show()

In [ ]:
part_id_plot_data = (
    df_clean["part_id"]
    .value_counts(dropna=False)
    .rename_axis("part_id")
    .reset_index(name="row_count")
    .sort_values("part_id", na_position="last")
)
part_id_plot_data["part_id_label"] = part_id_plot_data["part_id"].astype("string")

fig, ax = plt.subplots(figsize=(14, 5))
sns.barplot(data=part_id_plot_data, x="part_id_label", y="row_count", color="#B279A2", ax=ax)
ax.set_title("Part ID Distribution After Cleaning")
ax.set_xlabel("Part ID")
ax.set_ylabel("Row count")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df_clean, x="gpa_points", bins=30, color="#4C78A8", ax=axes[0])
axes[0].set_title("GPA Points Distribution After Cleaning")
axes[0].set_xlabel("GPA points")
axes[0].set_ylabel("Row count")

sns.histplot(data=df_clean, x="end_agpa_points", bins=30, color="#F58518", ax=axes[1])
axes[1].set_title("End AGPA Points Distribution After Cleaning")
axes[1].set_xlabel("End AGPA points")
axes[1].set_ylabel("Row count")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.histplot(data=df_clean, x="semester_reg_credits", bins=30, color="#54A24B", ax=axes[0])
axes[0].set_title("Semester Registered Credits Distribution After Cleaning")
axes[0].set_xlabel("Semester registered credits")
axes[0].set_ylabel("Row count")

sns.histplot(data=df_clean, x="total_pass_credits", bins=30, color="#E45756", ax=axes[1])
axes[1].set_title("Total Passed Credits Distribution After Cleaning")
axes[1].set_xlabel("Total passed credits")
axes[1].set_ylabel("Row count")

plt.tight_layout()
plt.show()

## Final displays

The clean table is available as `df_clean`. Audit reports are available as `dropped_permanent_status_report`, `dropped_degree_null_report`, `dropped_withdrawn_single_zero_activity_report`, and `dropped_core_null_report`.

In [ ]:
display(df_clean.head())

df_clean.info()

display(dropped_permanent_status_report.head())
display(dropped_degree_null_report.head())
display(dropped_withdrawn_single_zero_activity_report.head())
display(dropped_core_null_report.head())

In [ ]:
df_clean.to_parquet(r'D:\AI\Real projects\Academic_Advisor\data\preprocessed\v_add_student_degree_status_clean.parquet', index=False)

In [ ]:
df_clean['student_id'].nunique()

In [ ]:
df_clean[(df_clean['finish_status']=='WITHDRAWN')&(df_clean['gpa_points']==0)&(df_clean['prev_gpa_points']==0)]

In [ ]:
withdrawn_one_record_students = (
    df_clean[df_clean["finish_status"].eq("WITHDRAWN")]
    .groupby("student_id")
    .size()
)

num_withdrawn_students_with_one_record = (withdrawn_one_record_students == 1).sum()

print("WITHDRAWN students with exactly one record:", num_withdrawn_students_with_one_record)

## Drop students that exist in ADD_STUDENT_DEGREE_STATUS but do not exist in CRG_STUDENT_COURSE.
These students have no course history and are not useful for the academic recommendation/training dataset.
Create dropped_only_in_add_report.
Collect their student_status_id values.
## Drop rows using student_status_id, not DataFrame index.

In [ ]:
# ==============================
# Compare unique students between CRG_STUDENT_COURSE and ADD_STUDENT_DEGREE_STATUS
# ==============================
df_crg=pd.read_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_CRG_STUDENT_COURSE\clean_v_crg_student_course.parquet")
crg_students = set(df_crg["student_id"].dropna().unique())
add_students = set(df_clean["student_id"].dropna().unique())

students_in_both = crg_students & add_students
only_in_crg = crg_students - add_students
only_in_add = add_students - crg_students

print("========== STUDENT LEVEL OVERLAP ==========")
print("CRG unique students:", len(crg_students))
print("ADD unique students:", len(add_students))
print("Students in both:", len(students_in_both))
print("Only in CRG:", len(only_in_crg))
print("Only in ADD:", len(only_in_add))

print("\nCRG extra over ADD:", len(crg_students) - len(add_students))

In [ ]:
only_in_add_df = (
    df_clean[df_clean["student_id"].isin(only_in_add)]
    .sort_values("student_id")
)

display(only_in_add_df.head(100))

In [ ]:
only_in_add_df["finish_status"].value_counts(dropna=False)

In [ ]:
only_in_add_df['student_id'].nunique()

In [ ]:
# ============================================
# Drop students that exist only in ADD, not in CRG
# using student_status_id
# ============================================

# 1. Unique students in each table
crg_students = set(df_crg["student_id"].dropna().unique())
add_students = set(df_clean["student_id"].dropna().unique())

# 2. Students found only in ADD_STUDENT_DEGREE_STATUS
only_in_add_students = add_students - crg_students

print("Students only in ADD:", len(only_in_add_students))

# 3. Rows in ADD that belong to those students
only_in_add_mask = df_clean["student_id"].isin(only_in_add_students)

dropped_only_in_add_report = df_clean.loc[only_in_add_mask].copy()

print("Rows to drop only-in-ADD:", len(dropped_only_in_add_report))
print("Unique students to drop only-in-ADD:", dropped_only_in_add_report["student_id"].nunique())

print("\nfinish_status distribution:")
print(dropped_only_in_add_report["finish_status"].value_counts(dropna=False))

# 4. Save stable row IDs to drop
only_in_add_student_status_ids_to_drop = (
    dropped_only_in_add_report["student_status_id"]
    .dropna()
    .unique()
)

print("student_status_id values to drop:", len(only_in_add_student_status_ids_to_drop))

# 5. Drop using student_status_id, not index
df_add_clean_step = df_clean.loc[
    ~df_clean["student_status_id"].isin(only_in_add_student_status_ids_to_drop)
].copy()

print("Shape before:", df_clean.shape)
print("Shape after:", df_add_clean_step.shape)

# 6. Validation
assert not df_add_clean_step["student_status_id"].isin(only_in_add_student_status_ids_to_drop).any()

In [ ]:
display(
    dropped_only_in_add_report
    .groupby("finish_status", dropna=False)
    .agg(
        rows=("student_id", "size"),
        unique_students=("student_id", "nunique"),
        total_reg_courses_sum=("total_reg_courses", "sum"),
        total_reg_credits_sum=("total_reg_credits", "sum"),
        total_pass_courses_sum=("total_pass_courses", "sum"),
        total_pass_credits_sum=("total_pass_credits", "sum"),
        total_fail_courses_sum=("total_fail_courses", "sum"),
        total_fail_credits_sum=("total_fail_credits", "sum"),
    )
)

In [ ]:
df_clean['student_id'].nunique()

In [ ]:
# ============================================================
# Drop students existing only in ADD_STUDENT_DEGREE_STATUS clean table
# and not existing in CRG_STUDENT_COURSE
# Drop by student_status_id, not by index
# ============================================================

# 1. Get unique students from each table
crg_students = set(df_crg["student_id"].dropna().unique())
add_students = set(df_clean["student_id"].dropna().unique())

# 2. Students found in ADD/status table but not in CRG/course table
only_in_add_students = add_students - crg_students

print("Unique students in CRG:", len(crg_students))
print("Unique students in df_clean:", len(add_students))
print("Students only in df_clean / ADD:", len(only_in_add_students))

# 3. Find rows in df_clean that belong to only-in-ADD students
only_in_add_mask = df_clean["student_id"].isin(only_in_add_students)

# 4. Save dropped rows in a report before deleting
dropped_only_in_add_report = df_clean.loc[only_in_add_mask].copy()

print("Rows to drop:", len(dropped_only_in_add_report))
print("Unique students to drop:", dropped_only_in_add_report["student_id"].nunique())

print("\nfinish_status distribution of dropped rows:")
print(dropped_only_in_add_report["finish_status"].value_counts(dropna=False))

# 5. Collect stable primary keys to drop
student_status_ids_to_drop = (
    dropped_only_in_add_report["student_status_id"]
    .dropna()
    .unique()
)

print("\nstudent_status_id rows to drop:", len(student_status_ids_to_drop))

# 6. Drop using student_status_id
df_clean = df_clean.loc[
    ~df_clean["student_status_id"].isin(student_status_ids_to_drop)
].copy()

# 7. Validation
assert not df_clean["student_status_id"].isin(student_status_ids_to_drop).any(), \
    "Some dropped student_status_id values still exist in df_clean."

print("\nFinal df_clean shape:", df_clean.shape)
print("Remaining unique students:", df_clean["student_id"].nunique())

In [ ]:
# df_clean.to_parquet(r"D:\AI\Real projects\Academic_Advisor\data\preprocessed\V_ADD_STUDENT_DEGREE_STATUS\clean_v_add_student_degree_status.parquet", index=False)

In [ ]:
df_clean['student_id'].nunique()

In [ ]:
df_clean.info()